In [1]:
import sys
print(sys.executable)


/home/aytu/apps/docsearch/.venv/bin/python


In [4]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [5]:
import os
from openai import OpenAI

client = OpenAI(api_key=api_key)
resp = client.responses.create(
    model="gpt-5.4-mini",
    input="Reply with: notebook setup works"
)
print(resp.output_text)

notebook setup works


In [6]:
import nest_asyncio
nest_asyncio.apply()

In [7]:
from pydantic_ai import Agent

agent = Agent(
    "openai:gpt-5.4-mini",
    system_prompt="You are a concise assistant."
)

/home/aytu/apps/docsearch/.venv/lib/python3.11/site-packages/pydantic_ai/agent/__init__.py:410: PydanticAIDeprecationWarning: In v2.0, 'openai:' will resolve to the OpenAI Responses API by default. Use 'openai-chat:' to keep current Chat Completions behavior, or 'openai-responses:' to opt in early.
  self._model = models.infer_model(model)


In [8]:
result1 = agent.run_sync("My name is Ahmed.")
print(result1.output)

Nice to meet you, Ahmed.


In [9]:
import pandas as pd
pd.read_excel("/mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx")

,company name,alternate name,notes,link
0,ccc,ccc,"Test notes for company_1, variant 1",/data/docs/second_index.xlsx
1,ccc,ccc,"Test notes for company_1, variant 2",/data/docs/second_index.xlsx
2,ccc,ccc,"Test notes for company_1, variant 3",/data/docs/second_index.xlsx
3,company_2,company_2_2,"Test notes for company_2, variant 2",/data/docs/company_2_2.docx
4,company_2,company_2_3,"Test notes for company_2, variant 3",/data/docs/company_2_3.docx
...,...,...,...,...
992,company_200,company_200_1,"Test notes for company_200, variant 1",/data/docs/company_200_1.docx
993,company_200,company_200_2,"Test notes for company_200, variant 2",/data/docs/company_200_2.docx
994,company_200,company_200_3,"Test notes for company_200, variant 3",/data/docs/company_200_3.docx
995,company_200,company_200_4,"Test notes for company_200, variant 4",/data/docs/company_200_4.docx


In [10]:
from pydantic import BaseModel
from typing import Optional, Literal

class WorkflowState(BaseModel):
    conversation_id: str
    original_user_query: str
    current_index_path: str
    current_level: int = 0

    awaiting_user_clarification: bool = False
    clarification_question: Optional[str] = None

    selected_company_name: Optional[str] = None
    selected_alternate_name: Optional[str] = None
    selected_link: Optional[str] = None

    status: Literal[
        "searching_index",
        "needs_clarification",
        "reading_sop",
        "resolved",
        "error"
    ] = "searching_index"

In [11]:
class IndexRow(BaseModel):
    row_id: int
    company_name: str
    alternate_name: str | None = None
    link: str

In [12]:
from typing import Literal
from pydantic import BaseModel

class RouteDecision(BaseModel):
    action: Literal["select_row", "ask_user", "no_match"]
    selected_row_id: int | None = None
    question: str | None = None
    reasoning_short: str | None = None

In [13]:
ROOT_INDEX_PATH = "/mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx"

In [14]:
from pathlib import Path
import re

def normalize_path(raw_path: str) -> str:
    raw_path = raw_path.strip()

    if re.match(r"^[A-Za-z]:\\\\", raw_path):
        drive = raw_path[0].lower()
        rest = raw_path[2:].replace("\\\\", "/")
        return f"/mnt/{drive}{rest}"

    return str(Path(raw_path))

In [15]:
from pydantic import BaseModel, ConfigDict
from typing import Literal, Any

class ChatTurnResult(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    event_type: Literal["ask_user", "final_answer", "error"]
    assistant_text: str
    state: WorkflowState
    raw_result: Any = None

In [16]:
def handle_turn(user_input: str, state: WorkflowState | None) -> ChatTurnResult:
    if state is None:
        state = WorkflowState(
            conversation_id=new_id(),
            original_user_query=user_input,
            current_index_path=ROOT_INDEX_PATH,
            status="searching_index",
        )
    else:
        if state.awaiting_user_clarification:
            state.original_user_query += f"\nUser clarification: {user_input}"
            state.awaiting_user_clarification = False
            state.clarification_question = None

    while True:
        if state.status == "searching_index":
            rows = load_index_rows(state.current_index_path)

            decision = match_agent.run(
                user_query=state.original_user_query,
                rows=rows
            )

            if decision.action == "ask_user":
                state.awaiting_user_clarification = True
                state.clarification_question = decision.question
                state.status = "needs_clarification"
                return ask_user_result(decision.question, state)

            if decision.action == "no_match":
                state.status = "error"
                return error_result("Could not find a matching company.", state)

            selected_row = get_row_by_id(rows, decision.selected_row_id)
            selected_link = normalize_path(selected_row.link)

            state.selected_company_name = selected_row.company_name
            state.selected_alternate_name = selected_row.alternate_name
            state.selected_link = selected_link

            if selected_link.endswith(".docx"):
                state.status = "reading_sop"
                continue

            elif selected_link.endswith(".xlsx"):
                state.current_index_path = selected_link
                state.status = "searching_index"
                continue

            else:
                state.status = "error"
                return error_result("Unsupported linked file type.", state)

        elif state.status == "reading_sop":
            sop_text = parse_docx(state.selected_link)

            answer = qa_agent.run(
                user_query=state.original_user_query,
                sop_text=sop_text,
                company_name=state.selected_company_name,
            )

            state.status = "resolved"
            return final_answer_result(answer, state)

In [17]:
from pydantic import BaseModel
from pathlib import Path
import pandas as pd
import re

class IndexRow(BaseModel):
    row_id: int
    company_name: str
    alternate_name: str | None
    link: str

def normalize_windows_path(raw_path: str) -> str:
    raw_path = raw_path.strip()
    if re.match(r"^[A-Za-z]:\\", raw_path):
        drive = raw_path[0].lower()
        rest = raw_path[2:].replace("\\", "/")
        return f"/mnt/{drive}{rest}"
    return raw_path

def load_index_rows(index_path: str) -> list[IndexRow]:
    df = pd.read_excel(index_path) if index_path.endswith(".xlsx") else pd.read_csv(index_path)
    df = df.reset_index(drop=True)
    rows = []
    for i, row in df.iterrows():
        rows.append(
            IndexRow(
                row_id=i,
                company_name=str(row["company name"]).strip(),
                alternate_name=str(row["alternate name"]).strip() if pd.notna(row["alternate name"]) else None,
                link=str(row["link"]).strip(),
            )
        )
    return rows

In [18]:
from typing import Literal
from pydantic_ai import Agent

class RouteDecision(BaseModel):
    action: Literal["select_row", "ask_user", "no_match"]
    selected_row_id: int | None = None
    question: str | None = None

match_agent = Agent(
    "openai:gpt-4.1-mini",
    output_type=RouteDecision,
    system_prompt=(
        "You identify the best company row from the provided index rows. "
        "If multiple rows are plausible, ask exactly one short clarification question. "
        "If one row is clearly correct, return select_row."
    ),
)

/home/aytu/apps/docsearch/.venv/lib/python3.11/site-packages/pydantic_ai/agent/__init__.py:410: PydanticAIDeprecationWarning: In v2.0, 'openai:' will resolve to the OpenAI Responses API by default. Use 'openai-chat:' to keep current Chat Completions behavior, or 'openai-responses:' to opt in early.
  self._model = models.infer_model(model)


In [36]:
def build_match_prompt(user_input: str, rows: list[IndexRow]) -> str:
    row_text = "\n".join(
        f"row_id={r.row_id} | company={r.company_name} | alternate={r.alternate_name} | link={r.link}"
        for r in rows
    )
    return f"""
User query: {user_input}

Index rows:
{row_text}

Choose one:
- select_row
- ask_user
- no_match
"""

# async def handle_turn(user_input: str, state: WorkflowState | None) -> ChatTurnResult:
    if state is None:
        state = WorkflowState(
            conversation_id="demo-1",
            original_user_query=user_input,
            current_index_path=ROOT_INDEX_PATH,
            status="searching_index",
        )
    elif state.awaiting_user_clarification:
        state.original_user_query += f"\nUser clarification: {user_input}"
        state.awaiting_user_clarification = False
        state.clarification_question = None

    while True:
        rows = load_index_rows(state.current_index_path)
        table_text = dataframe_to_row_text(df)

        prompt = f"""
        User query:
        {user_input}

        Current table:
        {table_text}

        Your task:
        - Choose the best matching row if clear.
        - If ambiguous, ask one short clarification question.
        - If no row matches, return no_match.
        """
        async def handle_turn(user_input: str, state: WorkflowState | None) -> ChatTurnResult:
            if state is None:
                state = WorkflowState(
                    conversation_id="demo-1",
                    original_user_query=user_input,
                    current_index_path=ROOT_INDEX_PATH,
                    status="searching_index",
                )
            elif state.awaiting_user_clarification:
                state.original_user_query += f"\nUser clarification: {user_input}"
                state.awaiting_user_clarification = False
                state.clarification_question = None

            while True:
                rows = load_index_rows(state.current_index_path)
                table_text = dataframe_to_row_text(df)

                prompt = f"""
                User query:
                {user_input}

                Current table:
                {table_text}

                Your task:
                - Choose the best matching row if clear.
                - If ambiguous, ask one short clarification question.
                - If no row matches, return no_match.
                """

                decision = (
                    await match_agent.run(build_match_prompt(state.original_user_query, rows))
                ).output

                if decision.action == "ask_user":
                    state.awaiting_user_clarification = True
                    state.clarification_question = decision.question
                    state.status = "needs_clarification"
                    return ChatTurnResult(
                        event_type="ask_user",
                        assistant_text=decision.question,
                        state=state,
                    )

                if decision.action == "no_match":
                    state.status = "error"
                    return ChatTurnResult(
                        event_type="error",
                        assistant_text="I could not identify the company from the index.",
                        state=state,
                    )

                selected = next(
                    row for row in rows
                    if row.row_id == decision.selected_row_id
                )
                resolved_link = normalize_windows_path(selected.link)

                if resolved_link.lower().endswith(".docx"):
                    state.selected_company_name = selected.company_name
                    state.selected_link = resolved_link
                    state.status = "reading_sop"

                    sop_result = await answer_from_sop(
                        state.original_user_query,
                        state.selected_link,
                    )
                    state.status = "resolved"

                    return ChatTurnResult(
                        event_type="final_answer",
                        assistant_text=sop_result.answer,
                        state=state,
                    )

                if resolved_link.lower().endswith(".xlsx"):
                    state.current_index_path = resolved_link
                    state.status = "searching_index"
                    continue

                state.status = "error"
                return ChatTurnResult(
                    event_type="error",
                    assistant_text="The selected link is neither a DOCX SOP nor an XLSX index.",
                    state=state,
                )

        if decision.action == "ask_user":
            state.awaiting_user_clarification = True
            state.clarification_question = decision.question
            state.status = "needs_clarification"
            return ChatTurnResult(
                event_type="ask_user",
                assistant_text=decision.question,
                state=state,
            )

        if decision.action == "no_match":
            state.status = "error"
            return ChatTurnResult(
                event_type="error",
                assistant_text="I could not identify the company from the index.",
                state=state,
            )

        selected = next(r for r in rows if r.row_id == decision.selected_row_id)
        resolved_link = normalize_windows_path(selected.link)

        if resolved_link.lower().endswith(".docx"):
            state.selected_company_name = selected.company_name
            state.selected_link = resolved_link
            state.status = "reading_sop"

            sop_result = answer_from_sop(state.original_user_query, state.selected_link)
            state.status = "resolved"

            return ChatTurnResult(
                event_type="final_answer",
                assistant_text=sop_result.answer,
                state=state,
            )

        if resolved_link.lower().endswith(".xlsx"):
            state.current_index_path = resolved_link
            state.status = "searching_index"
            continue

        state.status = "error"
        return ChatTurnResult(
            event_type="error",
            assistant_text="The selected link is neither a DOCX SOP nor an XLSX index.",
            state=state,
        )

    return ChatTurnResult(
        event_type="final_answer",
        assistant_text=f"Resolved SOP path: {state.selected_link}",
        state=state,
    )

In [ ]:
# async def handle_turn(user_input: str, state: WorkflowState | None) -> ChatTurnResult:
#     if state is None:
#         state = WorkflowState(
#             conversation_id="demo-1",
#             original_user_query=user_input,
#             current_index_path=ROOT_INDEX_PATH,
#             status="searching_index",
#         )
#     elif state.awaiting_user_clarification:
#         state.original_user_query += f"\nUser clarification: {user_input}"
#         state.awaiting_user_clarification = False
#         state.clarification_question = None

#     while True:
#         df = load_small_table(state.current_index_path)
#         table_text = dataframe_to_row_text(df)

#         prompt = f"""
# User query:
# {state.original_user_query}

# Current table:
# {table_text}

# Your task:
# - Choose the best matching row if clear.
# - If ambiguous, ask one short clarification question.
# - If no row matches, return no_match.
# """

#         decision = (await match_agent.run(prompt)).output

#         if decision.action == "ask_user":
#             state.awaiting_user_clarification = True
#             state.clarification_question = decision.question
#             state.status = "needs_clarification"
#             return ChatTurnResult(
#                 event_type="ask_user",
#                 assistant_text=decision.question,
#                 state=state,
#                 raw_result=agent_result,
#             )

#         if decision.action == "no_match":
#             state.status = "error"
#             return ChatTurnResult(
#                 event_type="error",
#                 assistant_text="I could not identify the right row in the table.",
#                 state=state,
#             )

#         selected_row = df.iloc[decision.selected_row_id]
#         link = extract_link_from_row(selected_row)
#         resolved_link = normalize_windows_path(link)

#         if resolved_link.lower().endswith(".xlsx"):
#             state.current_index_path = resolved_link
#             state.status = "searching_index"
#             continue

#         if resolved_link.lower().endswith(".docx"):
#             state.selected_link = resolved_link
#             state.status = "reading_sop"

#             sop_result = await answer_from_sop(state.original_user_query, state.selected_link)
#             state.status = "resolved"

#             return ChatTurnResult(
#                 event_type="final_answer",
#                 assistant_text=sop_result.answer,
#                 state=state,
#             )

#         state.status = "error"
#         return ChatTurnResult(
#             event_type="error",
#             assistant_text="Selected row did not contain a usable DOCX or XLSX link.",
#             state=state,
#         )

In [20]:
async def handle_turn(
    user_input: str,
    state: WorkflowState | None,
    message_history=None,
) -> ChatTurnResult:
    if message_history is None:
        message_history = []

    if state is None:
        state = WorkflowState(
            conversation_id="demo-1",
            original_user_query=user_input,
            current_index_path=ROOT_INDEX_PATH,
            status="searching_index",
        )
    elif state.awaiting_user_clarification:
        state.original_user_query += f"\nUser clarification: {user_input}"
        state.awaiting_user_clarification = False
        state.clarification_question = None
    else:
        state.original_user_query = user_input

    while True:
        df = load_small_table(state.current_index_path)
        table_text = dataframe_to_row_text(df)

        prompt = f"""
User query:
{state.original_user_query}

Current table:
{table_text}

Your task:
- Choose the best matching row if clear.
- If ambiguous, ask one short clarification question.
- If no row matches, return no_match.
"""

        agent_result = await match_agent.run(
            prompt,
            message_history=message_history,
        )
        decision = agent_result.output

        if decision.action == "ask_user":
            state.awaiting_user_clarification = True
            state.clarification_question = decision.question
            state.status = "needs_clarification"

            return ChatTurnResult(
                event_type="ask_user",
                assistant_text=decision.question,
                state=state,
                raw_result=agent_result,
            )

        if decision.action == "no_match":
            state.status = "error"

            return ChatTurnResult(
                event_type="error",
                assistant_text="I could not identify the right row in the table.",
                state=state,
                raw_result=agent_result,
            )

        selected_row = df.iloc[decision.selected_row_id]
        link = extract_link_from_row(selected_row)

        if not link:
            state.status = "error"
            return ChatTurnResult(
                event_type="error",
                assistant_text="The selected row did not contain a usable file link.",
                state=state,
                raw_result=agent_result,
            )

        resolved_link = normalize_windows_path(link)

        if resolved_link.lower().endswith(".xlsx"):
            state.current_index_path = resolved_link
            state.status = "searching_index"
            message_history = agent_result.all_messages()
            continue

        if resolved_link.lower().endswith(".docx"):
            state.selected_link = resolved_link
            state.status = "reading_sop"

            sop_result = await answer_from_sop(
                state.original_user_query,
                state.selected_link,
            )
            state.status = "resolved"

            return ChatTurnResult(
                event_type="final_answer",
                assistant_text=sop_result.answer,
                state=state,
                raw_result=agent_result,
            )

        state.status = "error"
        return ChatTurnResult(
            event_type="error",
            assistant_text="Selected row did not contain a usable DOCX or XLSX link.",
            state=state,
            raw_result=agent_result,
        )

In [21]:
class SopAnswer(BaseModel):
    answer: str

qa_agent = Agent(
    "openai:gpt-4.1-mini",
    output_type=SopAnswer,
    system_prompt=(
        "Answer the user's question only from the SOP text provided. "
        "If the SOP does not contain the answer, say that clearly."
    ),
)

In [22]:
def load_small_table(index_path: str):
    path = Path(index_path)

    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path, engine="openpyxl")
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    df.columns = [str(c).strip() for c in df.columns]
    return df

def dataframe_to_row_text(df: pd.DataFrame) -> str:
    lines = []
    for i, row in df.reset_index(drop=True).iterrows():
        cells = [f"{col}={row[col]}" for col in df.columns]
        lines.append(f"row_id={i} | " + " | ".join(cells))
    return "\n".join(lines)

In [23]:
class TableDecision(BaseModel):
    action: Literal["select_row", "ask_user", "no_match"]
    selected_row_id: int | None = None
    question: str | None = None

In [24]:
def extract_link_from_row(row: pd.Series) -> str | None:
    for value in row.values:
        if isinstance(value, str):
            v = value.strip()
            if v.lower().endswith(".docx") or v.lower().endswith(".xlsx"):
                return v
    return None

In [25]:
from docx import Document

def read_docx_text(path: str) -> str:
    doc = Document(path)
    return "\n".join(p.text for p in doc.paragraphs if p.text.strip())

In [26]:
from docx import Document
from pydantic import BaseModel
from pydantic_ai import Agent

def read_docx_text(path: str) -> str:
    doc = Document(path)
    return "\n".join(
        p.text.strip()
        for p in doc.paragraphs
        if p.text and p.text.strip()
    )

class SopAnswer(BaseModel):
    answer: str

qa_agent = Agent(
    "openai:gpt-4.1-mini",
    output_type=SopAnswer,
    system_prompt=(
        "Answer the user's question only from the SOP text provided. "
        "If the answer is not in the SOP, say that clearly."
    ),
)

async def answer_from_sop(user_question: str, sop_path: str):
    sop_text = read_docx_text(sop_path)

    prompt = f"""
User question:
{user_question}

SOP text:
{sop_text}
"""
    result = await qa_agent.run(prompt)
    return result.output

In [27]:
# --- test conversation cell ---

state = None

async def chat_once(user_text: str):
    global state
    result = await handle_turn(user_text, state)
    state = result.state

    print(f"\nUSER: {user_text}")
    print(f"BOT ({result.event_type}): {result.assistant_text}")
    print(f"STATUS: {state.status}")

    if getattr(state, "current_index_path", None):
        print(f"CURRENT INDEX: {state.current_index_path}")

    if getattr(state, "selected_link", None):
        print(f"SELECTED LINK: {state.selected_link}")

    return result

In [28]:
await chat_once("I need the SOP for mike")


USER: I need the SOP for mike
BOT (error): I could not identify the right row in the table.
STATUS: error
CURRENT INDEX: /mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx


ChatTurnResult(event_type='error', assistant_text='I could not identify the right row in the table.', state=WorkflowState(conversation_id='demo-1', original_user_query='I need the SOP for mike', current_index_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx', current_level=0, awaiting_user_clarification=False, clarification_question=None, selected_company_name=None, selected_alternate_name=None, selected_link=None, status='error'), raw_result=AgentRunResult(output=RouteDecision(action='no_match', selected_row_id=None, question=None)))

In [29]:
await chat_once("I need the SOP for ccc")


USER: I need the SOP for ccc
BOT (ask_user): There are multiple SOP variants available for company 'ccc'. Do you want the first, second, or third variant?
STATUS: needs_clarification
CURRENT INDEX: /mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx


ChatTurnResult(event_type='ask_user', assistant_text="There are multiple SOP variants available for company 'ccc'. Do you want the first, second, or third variant?", state=WorkflowState(conversation_id='demo-1', original_user_query='I need the SOP for ccc', current_index_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx', current_level=0, awaiting_user_clarification=True, clarification_question="There are multiple SOP variants available for company 'ccc'. Do you want the first, second, or third variant?", selected_company_name=None, selected_alternate_name=None, selected_link=None, status='needs_clarification'), raw_result=AgentRunResult(output=RouteDecision(action='ask_user', selected_row_id=None, question="There are multiple SOP variants available for company 'ccc'. Do you want the first, second, or third variant?")))

In [30]:
result = await chat_once("i know the address. would that help?")


USER: i know the address. would that help?
BOT (ask_user): Can you please provide the address you know for the company 'ccc' to narrow down the correct SOP row?
STATUS: needs_clarification
CURRENT INDEX: /mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx


In [31]:
result = await agent.run("Si know the address. would that help?", message_history=message_history)

# Python-native message objects
history = result.all_messages()
for msg in history:
    print(msg.role, ":", msg.content)

# JSON-serializable version (e.g., to store or inspect)
history_json = result.all_messages_json()
print(history_json)

NameError: name 'message_history' is not defined

In [32]:
message_history = []

async def ask(text: str):
    global message_history
    result = await agent.run(text, message_history=message_history)
    message_history = result.all_messages()
    print(result.output)
    return result

In [33]:
await ask("I need the SOP for ccc")

Which “CCC” do you mean?

It could refer to several things, for example:
- **Credit Card Center / Customer Care Center**
- **Clinical Coordinating Center**
- **Call Center Center**
- **Child Care Center**
- **Construction Control Committee**
- something else in your organization

If you want, send me:
1. what CCC stands for,
2. the industry/context,
3. who the SOP is for,

and I can draft the SOP right away.


AgentRunResult(output='Which “CCC” do you mean?\n\nIt could refer to several things, for example:\n- **Credit Card Center / Customer Care Center**\n- **Clinical Coordinating Center**\n- **Call Center Center**\n- **Child Care Center**\n- **Construction Control Committee**\n- something else in your organization\n\nIf you want, send me:\n1. what CCC stands for,\n2. the industry/context,\n3. who the SOP is for,\n\nand I can draft the SOP right away.')

In [ ]:
print(result.state.model_dump_json())

{"conversation_id":"demo-1","original_user_query":"I need the SOP for ccc\nUser clarification: i know the address. would that help?\nUser clarification: i know the address. would that help?","current_index_path":"/mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx","current_level":0,"awaiting_user_clarification":true,"clarification_question":"Can you please specify which alternate name for CCC you are referring to: Osmanli or Selcuklu?","selected_company_name":null,"selected_alternate_name":null,"selected_link":null,"status":"needs_clarification"}


In [ ]:
await chat_once("what does sop tell me")


USER: what does sop tell me
BOT (final_answer): The SOP for hot mike is to use the red cable.
STATUS: resolved
CURRENT INDEX: /mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx
SELECTED LINK: /mnt/c/Users/maytu/OneDrive/Desktop/docs/2.docx


ChatTurnResult(event_type='final_answer', assistant_text='The SOP for hot mike is to use the red cable.', state=WorkflowState(conversation_id='demo-1', original_user_query='I need the SOP for hot mike', current_index_path='/mnt/c/Users/maytu/OneDrive/Desktop/docs/index.xlsx', current_level=0, awaiting_user_clarification=False, clarification_question=None, selected_company_name='mike', selected_alternate_name=None, selected_link='/mnt/c/Users/maytu/OneDrive/Desktop/docs/2.docx', status='resolved'))

In [ ]:
state = None
message_history = []
last_result = None

async def chat_once(user_text: str):
    global state, message_history, last_result

    result = await handle_turn(
        user_text,
        state,
        message_history=message_history,
    )

    state = result.state
    last_result = result.raw_result

    if last_result is not None:
        message_history = last_result.all_messages()

    print(f"\nUSER: {user_text}")
    print(f"BOT ({result.event_type}): {result.assistant_text}")
    print(f"STATUS: {state.status}")
    print(f"HISTORY LEN: {len(message_history)}")

    return result



In [ ]:
prompt = "I need the SOP for ccc"

agent_result = await match_agent.run(prompt, message_history=message_history)
decision = agent_result.output

In [ ]:
await chat_once("I need the SOP for ccc")
await chat_once("selcuklu")
print(last_result.all_messages_json())



USER: I need the SOP for ccc
BOT (ask_user): Do you need the SOP for the 'osmanli' or 'selcuklu' alternate name of ccc?
STATUS: needs_clarification
HISTORY LEN: 3

USER: selcuklu
BOT (final_answer): The SOP for Selcuklu company is to use key 1071.
STATUS: resolved
HISTORY LEN: 8
b'[{"parts":[{"content":"You identify the best company row from the provided index rows. If multiple rows are plausible, ask exactly one short clarification question. If one row is clearly correct, return select_row.","timestamp":"2026-05-26T20:16:02.056231Z","dynamic_ref":null,"part_kind":"system-prompt"},{"content":"\\nUser query:\\nI need the SOP for ccc\\n\\nCurrent table:\\nrow_id=0 | company name=mike | alternate name=cold mike | link=C:\\\\Users\\\\maytu\\\\OneDrive\\\\Desktop\\\\docs\\\\1.docx\\nrow_id=1 | company name=mike | alternate name=hot mike | link=C:\\\\Users\\\\maytu\\\\OneDrive\\\\Desktop\\\\docs\\\\2.docx\\nrow_id=2 | company name=ccc | alternate name=osmanli | link=C:\\\\Users\\\\maytu\\\\

In [ ]:
import gradio as gr

def init_session():
    return {
        "workflow_state": None,
        "message_history": [],
    }

async def respond(message, chat_history, session):
    if not message or not message.strip():
        return chat_history, session, {}, ""

    if session is None:
        session = init_session()

    result = await handle_turn(
        user_input=message,
        state=session["workflow_state"],
        message_history=session["message_history"],
    )

    session["workflow_state"] = result.state

    if result.raw_result is not None:
        session["message_history"] = result.raw_result.all_messages()

    chat_history = chat_history or []
    chat_history = chat_history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": result.assistant_text},
    ]

    debug_payload = {
        "event_type": result.event_type,
        "assistant_text": result.assistant_text,
        "workflow_state": result.state.model_dump(),
        "message_count": len(session["message_history"]),
    }

    return chat_history, session, debug_payload, ""

def reset_session():
    return [], init_session(), {}

with gr.Blocks() as demo:
    gr.Markdown("## SOP Bot")

    chatbot = gr.Chatbot(height=500)
    msg = gr.Textbox(label="Ask about a company or SOP", placeholder="Ask a question...")
    send = gr.Button("Send")
    clear = gr.Button("Reset")
    session = gr.State(init_session())
    debug = gr.JSON(label="Debug State")

    send.click(
        respond,
        inputs=[msg, chatbot, session],
        outputs=[chatbot, session, debug, msg],
    )

    msg.submit(
        respond,
        inputs=[msg, chatbot, session],
        outputs=[chatbot, session, debug, msg],
    )

    clear.click(
        reset_session,
        inputs=[],
        outputs=[chatbot, session, debug],
    )

demo.launch()

/home/aytu/apps/docsearch/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
import gradio as gr
print(gr.__version__)

/home/aytu/apps/docsearch/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


6.14.0


## Evaluation harness

Run the offline regression cell without making model calls. First run the definition cells for the workflow types, helpers, and current `handle_turn`; do not use Run All, because earlier smoke-test/conversation cells can call the API and use local files. The optional live eval uses synthetic company/index/SOP data, calls the configured agents, and checks routing plus required answer evidence. Run it only when you intend to use API credits.

In [37]:
from unittest.mock import patch

class _OfflineAgentResult:
    def __init__(self, output):
        self.output = output

    def all_messages(self):
        return []

class _OfflineMatchAgent:
    def __init__(self, decisions):
        self.decisions = iter(decisions)
        self.calls = 0

    async def run(self, *args, **kwargs):
        self.calls += 1
        try:
            decision = next(self.decisions)
        except StopIteration as exc:
            raise AssertionError('Workflow made more routing calls than expected') from exc
        return _OfflineAgentResult(decision)

def _eval_table(rows):
    return pd.DataFrame(rows, columns=['company name', 'alternate name', 'link'])

async def run_offline_regression_suite():
    cases = [
        {
            'name': 'select_docx_and_answer',
            'tables': {'/eval/root.xlsx': _eval_table([('Acme', 'Acme North', '/eval/acme.docx')])},
            'decisions': [RouteDecision(action='select_row', selected_row_id=0)],
            'event': 'final_answer', 'status': 'resolved', 'link': '/eval/acme.docx',
        },
        {
            'name': 'ask_for_clarification',
            'tables': {'/eval/root.xlsx': _eval_table([('Acme', 'Acme North', '/eval/north.docx'), ('Acme', 'Acme South', '/eval/south.docx')])},
            'decisions': [RouteDecision(action='ask_user', question='North or South?')],
            'event': 'ask_user', 'status': 'needs_clarification', 'link': None,
        },
        {
            'name': 'return_no_match',
            'tables': {'/eval/root.xlsx': _eval_table([('Acme', None, '/eval/acme.docx')])},
            'decisions': [RouteDecision(action='no_match')],
            'event': 'error', 'status': 'error', 'link': None,
        },
        {
            'name': 'follow_nested_index',
            'tables': {
                '/eval/root.xlsx': _eval_table([('Acme', None, '/eval/nested.xlsx')]),
                '/eval/nested.xlsx': _eval_table([('Acme North', None, '/eval/north.docx')]),
            },
            'decisions': [RouteDecision(action='select_row', selected_row_id=0), RouteDecision(action='select_row', selected_row_id=0)],
            'event': 'final_answer', 'status': 'resolved', 'link': '/eval/north.docx',
        },
    ]
    reports = []
    for case in cases:
        fake_match_agent = _OfflineMatchAgent(case['decisions'])

        def fake_load_table(path):
            if path not in case['tables']:
                raise AssertionError(f'Unexpected index path: {path}')
            return case['tables'][path].copy()

        async def fake_answer_from_sop(user_question, sop_path):
            return SopAnswer(answer='Escalate within 2 business hours.')

        with patch.dict(globals(), {
            'ROOT_INDEX_PATH': '/eval/root.xlsx',
            'load_small_table': fake_load_table,
            'match_agent': fake_match_agent,
            'answer_from_sop': fake_answer_from_sop,
        }):
            result = await handle_turn('How should I handle this?', None)

        assert result.event_type == case['event'], (case['name'], result.event_type)
        assert result.state.status == case['status'], (case['name'], result.state.status)
        assert result.state.selected_link == case['link'], (case['name'], result.state.selected_link)
        assert fake_match_agent.calls == len(case['decisions']), case['name']
        if case['name'] == 'select_docx_and_answer':
            assert '2 business hours' in result.assistant_text
        reports.append({'case': case['name'], 'passed': True, 'event': result.event_type})
    return pd.DataFrame(reports)

offline_results = await run_offline_regression_suite()
display(offline_results)

,case,passed,event
0,select_docx_and_answer,True,final_answer
1,ask_for_clarification,True,ask_user
2,return_no_match,True,error
3,follow_nested_index,True,final_answer


In [39]:
LIVE_EVAL_CASES = [
    {
        'name': 'acme_escalation',
        'query': 'How quickly should an Acme issue be escalated?',
        'rows': [('Acme', 'Acme North', '/eval/acme.docx')],
        'sop_text': 'Escalate any unresolved issue to the on-call lead within 2 business hours.',
        'expected_link': '/eval/acme.docx',
        'required_answer_terms': ['2', 'business hours'],
    },
]

def _normalize_eval_text(text):
    return re.sub(r'\s+', ' ', text.casefold().replace('two', '2')).strip()

async def run_live_eval(cases=LIVE_EVAL_CASES):
    if not os.getenv('OPENAI_API_KEY'):
        raise RuntimeError('Set OPENAI_API_KEY before running live model evaluations.')

    reports = []
    for case in cases:
        fixture_table = _eval_table(case['rows'])
        with patch.dict(globals(), {
            'ROOT_INDEX_PATH': '/eval/live-root.xlsx',
            'load_small_table': lambda path: fixture_table.copy(),
            'read_docx_text': lambda path: case['sop_text'],
        }):
            result = await handle_turn(case['query'], None)

        normalized_answer = _normalize_eval_text(result.assistant_text)
        route_correct = result.state.selected_link == case['expected_link']
        answer_evidence_present = all(
            _normalize_eval_text(term) in normalized_answer
            for term in case['required_answer_terms']
        )
        reports.append({
            'case': case['name'],
            'event': result.event_type,
            'route_correct': route_correct,
            'answer_evidence_present': answer_evidence_present,
            'passed': route_correct and answer_evidence_present,
            'answer': result.assistant_text,
        })
    return pd.DataFrame(reports)

# Opt in explicitly; this makes live model calls and may incur API charges.
live_results = await run_live_eval()
display(live_results)

,case,event,route_correct,answer_evidence_present,passed,answer
0,acme_escalation,final_answer,True,True,True,An Acme issue should be escalated to the on-ca...
